<a href="https://colab.research.google.com/github/Noman654/dataengineer_prep/blob/test/03_pyspark/03_joins.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔗 Joins — The Loyalty Reconciliation Nightmare

---

### 💬 New message from Dev (Store Ops Lead)

> **Dev** — Thursday 4:47 PM
>
> okay i need your help. the loyalty ↔ POS reconciliation is **completely broken** and marcus is going to fire me if we don't fix it before friday's board meeting.
>
> three things going wrong:
>
> 1. the revenue report shows **zero loyalty redemptions matched to transactions**. literally zero. finance thinks there's a huge bug. (i looked at the data and there ARE redemptions in both tables, they're just not joining.)
>
> 2. even once we fix that — we're opening store 104 as a new franchise location and they're complaining that their loyalty sign-up rate looks like 0%. i suspect the integration is broken for them specifically but i need to PROVE it with data.
>
> 3. the reconciliation job takes **40 minutes** on the real data. we used to run it in 3. something changed after we onboarded the new franchise locations last quarter.
>
> can you debug this and get it running under 5 min? send me the working code by EOD.

---

You just opened your laptop and Slack has 47 unread messages from Dev. Let's go.

## 🎯 What you'll learn

- All the join types that matter in PySpark: `inner`, `left`, `right`, `full_outer`, `left_semi`, `left_anti`
- The classic **type-mismatch bug** — when `join()` silently returns empty and you lose an afternoon to it
- **Broadcast joins** — the #1 performance trick you'll use every week
- **Skew detection** and handling — the single biggest reason joins go slow in production
- How to read `.explain()` output to verify your join strategy actually got applied

**Prerequisites:** You're comfortable with DataFrames, `filter`, `groupBy`. You've seen `join()` syntax before.

**Difficulty:** 🟡 Intermediate (with ⚡ touches in the skew and AQE sections)

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import broadcast

spark = (
    SparkSession.builder
    .appName("zephyr_joins")
    # Enable AQE — Spark 3's adaptive query execution handles skew automatically.
    # We'll see it in action later in this notebook.
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.skewJoin.enabled", "true")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

## 📦 The data

You have two tables from Zephyr's systems:

- **`pos_transactions`** — every sale, logged by the POS terminals. `store_id` is an **int** (42, 101, 102, 103, 104).
- **`loyalty_redemptions`** — every time a customer redeemed points, logged by the loyalty app. `store_id` is a **string** (`"store_042"`, `"store_101"`, ...).

Notice the type mismatch already? Good. That's going to be Problem 1 — but don't jump ahead, let's see how Spark "helps" us discover it.

We've deliberately built the sample to mirror the real bugs Dev is seeing:
- Store 42 has **way more traffic** than any other store (it's the downtown flagship) → future skew
- Store 104 (new franchise) has POS transactions but **no loyalty records** → loyalty integration broken
- The small stores (101/102/103) have both POS and loyalty → these should reconcile cleanly

*In the real repo, this would load from `../assets/sample_data/`. Here we generate inline so the notebook is self-contained.*

In [ ]:
# pos_transactions: store_id is INTEGER
# Store 42 is heavily over-represented — that's our soon-to-be-skewed key.
pos_data = [
    # store 42 (downtown flagship) — 10 transactions, the hot key
    ("tx_001", 42,  "c_alice",   12.50),
    ("tx_002", 42,  "c_bob",      8.75),
    ("tx_003", 42,  "c_carol",   15.00),
    ("tx_004", 42,  "c_dave",     9.25),
    ("tx_005", 42,  "c_eve",     11.00),
    ("tx_006", 42,  "c_frank",    7.50),
    ("tx_007", 42,  "c_gina",    13.25),
    ("tx_008", 42,  "c_harry",   10.00),
    ("tx_009", 42,  "c_isla",    14.75),
    ("tx_010", 42,  "c_jack",     6.50),
    # store 101 — 3 transactions
    ("tx_011", 101, "c_kate",     9.00),
    ("tx_012", 101, "c_liam",    11.50),
    ("tx_013", 101, "c_mia",      8.25),
    # store 102 — 3 transactions
    ("tx_014", 102, "c_nick",    10.75),
    ("tx_015", 102, "c_olivia",  12.00),
    ("tx_016", 102, "c_pete",     7.25),
    # store 103 — 2 transactions
    ("tx_017", 103, "c_quinn",    9.50),
    ("tx_018", 103, "c_rose",    13.00),
    # store 104 — NEW FRANCHISE, 2 transactions, NO loyalty records
    ("tx_019", 104, "c_sam",     15.50),
    ("tx_020", 104, "c_tina",    11.75),
]
pos_transactions = spark.createDataFrame(
    pos_data,
    ["tx_id", "store_id", "customer_id", "amount"],
)

# loyalty_redemptions: store_id is STRING (zero-padded format from the app)
loyalty_data = [
    # "store_042" — 8 redemptions (hot key on this side too)
    ("lr_001", "store_042", "c_alice",   50),
    ("lr_002", "store_042", "c_bob",     30),
    ("lr_003", "store_042", "c_carol",   75),
    ("lr_004", "store_042", "c_dave",    40),
    ("lr_005", "store_042", "c_eve",     60),
    ("lr_006", "store_042", "c_frank",   25),
    ("lr_007", "store_042", "c_gina",    55),
    ("lr_008", "store_042", "c_harry",   35),
    # "store_101" — 2 redemptions
    ("lr_009", "store_101", "c_kate",    20),
    ("lr_010", "store_101", "c_liam",    45),
    # "store_102" — 2 redemptions
    ("lr_011", "store_102", "c_nick",    30),
    ("lr_012", "store_102", "c_olivia",  40),
    # "store_103" — 1 redemption
    ("lr_013", "store_103", "c_quinn",   15),
    # NOTE: store_104 is DELIBERATELY absent (new franchise, loyalty broken)
]
loyalty_redemptions = spark.createDataFrame(
    loyalty_data,
    ["redemption_id", "store_id", "customer_id", "points_redeemed"],
)

### 🔍 Inspect both tables

Always look at your data before joining it. Schemas first (this is where the bug will jump out if you're paying attention) — then a sample.

In [ ]:
print("=== pos_transactions ===")
pos_transactions.printSchema()
pos_transactions.show(5)
print(f"Total rows: {pos_transactions.count()}")

print("\n=== loyalty_redemptions ===")
loyalty_redemptions.printSchema()
loyalty_redemptions.show(5)
print(f"Total rows: {loyalty_redemptions.count()}")

## 🚨 Problem 1 — the join returns nothing

You spotted the bug in the schemas, right? `pos_transactions.store_id` is **`long`** (integer), `loyalty_redemptions.store_id` is **`string`**. Let's see what Spark does when we naively join them.

In [ ]:
broken = pos_transactions.join(
    loyalty_redemptions,
    on="store_id",
    how="inner",
)

print(f"Joined row count: {broken.count()}")
broken.show()

### Why that returned zero

Spark couldn't join on `store_id` at all because one column is `long` and the other is `string`. Depending on your Spark version you either get:
- A silent **empty result** (yikes), or
- An `AnalysisException` complaining about incompatible types

Either way: **the loyalty app logs `"store_042"` and POS logs `42`. These will never hash-equal.**

This is the single most common "I lost two hours to a bug" moment in PySpark. Always check your schemas before joining.

### The fix — cast one side to match the other

Two options: cast `pos.store_id` to string, or cast `loyalty.store_id` to int. **Cast the smaller side** (less data to rewrite). Here, loyalty has 13 rows vs pos with 20 — close call. Let's normalize loyalty to int by stripping the `"store_"` prefix.

In [ ]:
# Normalize loyalty.store_id: "store_042" -> 42
loyalty_normalized = loyalty_redemptions.withColumn(
    "store_id",
    F.regexp_replace("store_id", "store_0*", "").cast("int"),
)

# Now the join works
matched = pos_transactions.join(
    loyalty_normalized,
    on="store_id",
    how="inner",
)

print(f"Matched rows: {matched.count()}")
matched.select("store_id", "tx_id", "redemption_id", "amount", "points_redeemed").show()

## 🚨 Problem 2 — Dev's other concern: the new franchise

Great, the inner join works. But Dev specifically said:

> *"store 104 (new franchise) is complaining that their loyalty sign-up rate looks like 0%. i suspect the integration is broken for them specifically but i need to PROVE it with data."*

An **inner join** silently hides this problem — store 104 has POS transactions but no loyalty records, so it just disappears from the joined result. We need to find POS transactions where **no loyalty record exists** at that store.

This is the job of a **LEFT ANTI JOIN**.

### The join types cheat sheet (memorize this)

| Type | Returns |
|---|---|
| `inner` | Only rows with matches on BOTH sides |
| `left` | All left rows, matching right rows (nulls where unmatched) |
| `right` | All right rows, matching left rows (nulls where unmatched) |
| `full_outer` | All rows from both sides (nulls where unmatched) |
| `left_semi` | Left rows that have a match (but only the LEFT columns — no right-side data) |
| `left_anti` | Left rows that have **no** match. Like a "NOT EXISTS". |

**`left_anti` is the one people forget exists** — and it's exactly what we need here. We want: "POS transactions where no corresponding loyalty store exists."

The mental model: `LEFT ANTI JOIN` is SQL's `WHERE NOT EXISTS`, but implemented as a set operation instead of a correlated subquery. It's fast and clean.

In [ ]:
# Find POS transactions whose store has NO loyalty records
orphan_pos = pos_transactions.join(
    loyalty_normalized,
    on="store_id",
    how="left_anti",
)

print(f"Orphan POS rows (stores with no loyalty integration): {orphan_pos.count()}")
orphan_pos.show()

# Confirm which stores are affected
print("Affected stores:")
orphan_pos.select("store_id").distinct().show()

## 🚨 Problem 3 — the job takes 40 minutes

Great — we've fixed correctness. Now for speed.

On our 20-row sample, everything is instant. But Dev said the real job takes 40 minutes. Why?

On real-world data, the standard join strategy is a **shuffle hash join**: both sides get repartitioned by the join key across all executors, then matched up. That shuffle is expensive — especially if one side is **small**, because you're moving data around pointlessly.

### The trick: broadcast the small side

If one side fits in memory, Spark can **broadcast** it — send a full copy to every executor — and then every task can look up the other side locally, **no shuffle at all**. This is called a **broadcast hash join**, and it's the #1 performance trick in PySpark.

**When to use it:** one side is under ~10 MB (default `spark.sql.autoBroadcastJoinThreshold`). Spark will *sometimes* auto-broadcast based on size estimates, but those estimates are often wrong — it's safer to be explicit with a `broadcast()` hint.

Let's check the sizes of our two tables and broadcast the small one.

In [ ]:
# Size check (rough)
print(f"pos_transactions: {pos_transactions.count()} rows")
print(f"loyalty_normalized: {loyalty_normalized.count()} rows")

# loyalty is the smaller side — broadcast it
result = pos_transactions.join(
    broadcast(loyalty_normalized),
    on="store_id",
    how="inner",
)

# Confirm the plan actually used broadcast — look for "BroadcastHashJoin" in the output
print("\n=== Query plan ===")
result.explain()

## 🚨 Problem 4 — store 42 is a bottleneck

Broadcast got us most of the way there. But Dev also mentioned:

> *"something changed after we onboarded the new franchise locations last quarter"*

What changed: now that there are more stores, but **store 42 (downtown flagship) still has most of the traffic**, any join or groupBy on `store_id` is going to be *very* unbalanced. One executor gets slammed with store 42's data while the others sit idle. That's **skew**.

In our sample, store 42 has 10 out of 20 transactions — 50% of the data on one key. In production that might be 100M out of 500M rows. The symptoms:
- A few tasks finish in seconds, one task takes forever
- In the Spark UI: **max task duration >> median task duration**
- Your whole job's runtime is bottlenecked on one skewed task

Let's detect it first.

In [ ]:
# Detect skew — count rows per join key and look for outliers
print("Distribution of store_id in pos_transactions:")
(pos_transactions
 .groupBy("store_id")
 .count()
 .orderBy(F.col("count").desc())
 .show())

# On real data you'd compute min/median/max and check the ratio
# Max should be within ~2-3x of median. If max is 10x+ median, you have skew.

### How to fix skew

Three levels of fix, in order of preference:

**1. AQE skew join (Spark 3+)** — **try this first.** We enabled it in the SparkSession at the top of this notebook. AQE detects skewed partitions at runtime and automatically splits them into smaller ones. Zero code changes. In many cases, this alone solves the problem.

**2. Broadcast the small side** — which we already did. If both sides are "small enough", this is always better than any shuffle strategy.

**3. Manual salting** — if the skewed side is huge AND AQE isn't enough AND broadcast isn't an option. The idea: add a random "salt" bucket to the hot key so its rows distribute across many partitions, then replicate the other side once per salt value.

Salting is the **nuclear option** — powerful, but adds complexity. Save it for when the simpler fixes don't work. The Boss Level at the bottom of this notebook walks through it.

For the immediate fix Dev needs, **AQE + broadcast is already enough** for our data. Let's confirm the final working query.

## 📤 Ship it to Dev

```python
# The reconciled revenue + loyalty view — what Dev actually asked for
reconciled = pos_transactions.join(
    broadcast(loyalty_normalized),
    on="store_id",
    how="left",   # keep ALL POS transactions, including store 104 with no loyalty
)

# Two useful aggregates for Dev and finance:

# 1. Revenue + loyalty redemption by store (inner semantics)
by_store = (
    reconciled
    .groupBy("store_id")
    .agg(
        F.sum("amount").alias("total_revenue"),
        F.sum(F.coalesce("points_redeemed", F.lit(0))).alias("total_points_redeemed"),
        F.count("tx_id").alias("tx_count"),
        F.count("redemption_id").alias("redemption_count"),
    )
    .orderBy("store_id")
)
by_store.show()

# 2. Stores with POS but no loyalty integration (the orphans)
broken_integrations = (
    pos_transactions
    .join(loyalty_normalized, on="store_id", how="left_anti")
    .select("store_id").distinct()
)
broken_integrations.show()
```

> **You** → Dev: ok this should work. three fixes:
> 1. the store_id type mismatch was the cause of the zero-match bug. cast fixed it.
> 2. store 104 is DEFINITELY orphaned — zero loyalty records. here's the proof (left anti join).
> 3. added broadcast hint + AQE which should bring the job under 5 min. explain plan shows BroadcastHashJoin now instead of SortMergeJoin. 👍
>
> **Dev**: YOU ARE A LIFESAVER i'm sending you 4 cups of store 42 coffee

## 🎯 Your turn — Dev's follow-up

> **Dev** — Friday 9:02 AM
>
> ok the board meeting went great THANK YOU. but now marcus has a follow-up question and i need you again.
>
> he wants to know: **for each store, what percentage of transactions resulted in a loyalty redemption?** (redemptions / transactions). basically the "loyalty attachment rate" per store. stores with low rates are where we're leaving money on the table.
>
> same data. figure it out.

Try it in the cell below. Same join skills, but you'll need to combine a left join with an aggregation.

Hints if you're stuck:
1. `left` join (not inner) so stores with zero redemptions still appear
2. `groupBy("store_id")` then count both `tx_id` and `redemption_id`
3. Compute the ratio, be careful with nulls

Solution is at the bottom.

In [ ]:
# your code here



## 🏆 Boss Level — Marcus's fraud detection + manual skew handling

> **Marcus (CFO)** — Friday 5:50 PM (of course)
>
> I'm seeing loyalty redemptions in the data where the customer_id doesn't appear in ANY pos transaction at that store. That shouldn't be possible — you can't redeem points without buying something. Either our data is wrong or we have fraud.
>
> Find every `(store_id, customer_id)` pair where a loyalty redemption exists but there is NO corresponding POS transaction for that customer at that store. Be efficient — assume we're running this on 500M transactions and 50M redemptions.
>
> **Extra credit:** write the query in a way that handles skew manually (salting), and compare the `.explain()` output against the non-salted version.

This one teaches:
1. `left_anti` on a **composite key** (store + customer), not just one column
2. **Manual salting** — the nuclear option for skew when AQE + broadcast aren't enough
3. Reading `.explain()` output for `BroadcastHashJoin` vs `SortMergeJoin` vs `ShuffledHashJoin`

**Think before you code:** the key insight is that `left_anti` doesn't just have to be on one column — you can anti-join on a *combination*. And for the salting part: what do you add to each row of the big side, and what do you do to the small side to make the join still work?

In [ ]:
# your code here



---

## 💡 Solutions

> ⚠️ **Spoilers below.** Don't scroll any further until you've tried both exercises. Struggling with them for 15 minutes is worth more than reading the solution in 30 seconds.
>
> Still here? OK, let's walk through both.

### 💡 Solution 1 — loyalty attachment rate per store

**The ask:** for each store, what fraction of transactions resulted in a loyalty redemption?

**The pattern:**
1. `left` join POS → loyalty (keep all transactions, even unmatched ones)
2. `groupBy("store_id")`
3. Count `tx_id` (total transactions) and count `redemption_id` (only non-null = matched)
4. Divide: `redemptions / transactions`
5. Order by rate to find the worst-performing stores

**Why a LEFT join, not INNER?** If we used inner, store 104 (no loyalty) would disappear entirely — but that's exactly the store Dev wants to flag. Left keeps every POS transaction, and the redemption fields come back NULL for stores with no match. `count(col)` conveniently ignores NULLs, so the counts work out correctly.

In [ ]:
attachment_rate = (
    pos_transactions
    .join(broadcast(loyalty_normalized), on="store_id", how="left")
    .groupBy("store_id")
    .agg(
        F.count("tx_id").alias("tx_count"),
        F.count("redemption_id").alias("redemption_count"),  # count() ignores nulls
    )
    .withColumn(
        "attachment_rate",
        F.round(F.col("redemption_count") / F.col("tx_count"), 2)
    )
    .orderBy("attachment_rate")  # worst first
)

attachment_rate.show()

### 💡 Solution 2 — fraud detection + manual salting

**Part A — the composite-key anti-join**

A `left_anti` on a single column finds rows where no match exists for that column. But Marcus wants something more specific: *no match exists for the combination of `(store_id, customer_id)`*. You handle this the same way — just pass multiple columns to the `on` parameter.

**Part B — the salting pattern**

Salting breaks a hot key into N virtual sub-keys:

1. Add a random `salt` column to the big side (values 0 to N-1)
2. For the small side, explode each row into N copies — one per possible salt value
3. Join on `(original_key, salt)` — now the hot key's rows distribute across N partitions instead of 1

**The cost:** you 10x the small side (cheap if it's already small). **The benefit:** the skewed partition splits evenly.

For *our* data, this is overkill — we only have 20 rows and 4 stores. But the pattern matters. On real data with 500M rows and one hot key, salting can turn a 2-hour job into 5 minutes.

In [ ]:
# === Part A: composite-key left anti join ===
# "Redemptions where no POS transaction exists for that (store, customer)"
# This is the fraud / data-quality check.

# First, normalize loyalty.store_id to int as before
loyalty_norm = loyalty_redemptions.withColumn(
    "store_id",
    F.regexp_replace("store_id", "store_0*", "").cast("int"),
)

suspicious = loyalty_norm.join(
    pos_transactions,
    on=["store_id", "customer_id"],   # composite key — this is the key move
    how="left_anti",
)

print("Suspicious redemptions (no matching POS transaction):")
suspicious.show()
# In our sample all redemptions DO have matching POS, so this is empty.
# On real data, orphans here = either bad data or actual fraud.


# === Part B: manual salting (pattern demo) ===
# Suppose pos_transactions was so large that even AQE can't handle store 42's skew.
# We'd salt it manually.

N = 4  # number of salt buckets — tune based on skew severity

pos_salted = pos_transactions.withColumn(
    "salt",
    (F.rand() * N).cast("int")
)

# Small side: replicate N times, once per salt value
loyalty_exploded = (
    loyalty_norm
    .withColumn("salt", F.explode(F.array([F.lit(i) for i in range(N)])))
)

# Join on composite key (store_id, salt) — hot key now distributes across N partitions
salted_result = pos_salted.join(
    loyalty_exploded,
    on=["store_id", "salt"],
    how="inner",
).drop("salt")

print(f"Salted join result rows: {salted_result.count()}")
salted_result.select("store_id", "tx_id", "redemption_id").show()

# Compare the plans — salted version has a different partitioning strategy
print("\n=== Salted join query plan ===")
salted_result.explain()

## 📚 Further reading

- [Spark SQL join reference](https://spark.apache.org/docs/latest/sql-ref-syntax-qry-select-join.html) — the canonical doc for all join types
- [Spark Performance Tuning — Join hints](https://spark.apache.org/docs/latest/sql-performance-tuning.html#join-strategy-hints-for-sql-queries) — broadcast, merge, shuffle hash, shuffle replicate
- [Databricks: AQE deep dive](https://www.databricks.com/blog/2020/05/29/adaptive-query-execution-speeding-up-spark-sql-at-runtime.html) — the canonical AQE post
- [`theory/shuffle_and_partitioning.md`](theory/shuffle_and_partitioning.md) — the quick-review doc for shuffle internals, skew, and partitioning strategies. Read this before your next interview.

### What's next in this module
- `04_groupby_aggregations.ipynb` — groupBy, agg, grouping sets, rollups (🟢 basics)
- `05_null_handling_dedup.ipynb` — the 2023 Zephyr duplicate incident, safe dedup patterns (🟡)
- `06_nested_data.ipynb` — exploding Zephyr's loyalty event structs (🟡)

Save your notebook. Grab a coffee. Store 42's fresh.